# Go formatting verbs

In [1]:
func FmtTable(verbs []string, values []any) string {
	var b strings.Builder

	b.WriteString(`<table style="white-space: pre; font-family: monospace;">`)

	b.WriteString("<tbody>\n")
	for _, verb := range verbs {
		fmt.Fprintf(&b, "<tr><th>%s</th>", html.EscapeString(verb))
		for _, val := range values {
			result := fmt.Sprintf(verb, val)
            var cell string
			if strings.HasPrefix(result, "%!") {
				cell = `<span style="font-size: 70%">N/A</span>`
			} else {
                cell = html.EscapeString(result)
            }
			fmt.Fprintf(&b, "<td>%s</td>", cell)
		}
		fmt.Fprintf(&b, "<th>%s</th></tr>", html.EscapeString(verb))
	}
	b.WriteString("</tbody>\n</table>\n")

	return b.String()
}

In [2]:
type Student struct {
    Name   string
    Height float32
}

var ana = Student{"Ana", 1.67}
%%
fmt.Print(ana)

{Ana 1.67}

## General verbs

```
✓ %v	the value in a default format
✓    	when printing structs, the plus flag (%+v) adds field names
✓ %#v	a Go-syntax representation of the value
    	(floating-point infinities and NaNs print as ±Inf and NaN)
✓ %T	a Go-syntax representation of the type of the value
  %%	a literal percent sign; consumes no value
```

In [3]:
%%
verbs := []string{"%T", "%#v", "%v", "%+v", "%q", "%+q", "%x", "% x"}
values := []any{42, 'Ç', "olá\tmundo", []rune{'A', 'Á'}, Student{"Zé", 1.89}}
html := FmtTable(verbs, values)
gonbui.DisplayHTML(html)

%T,int,int32,string,[]int32,main.Student,%T
%#v,42,199,"""olá\tmundo""","[]int32{65, 193}","main.Student{Name:""Zé"", Height:1.89}",%#v
%v,42,199,olá mundo,[65 193],{Zé 1.89},%v
%+v,42,199,olá mundo,[65 193],{Name:Zé Height:1.89},%+v
%q,'*','Ç',"""olá\tmundo""",['A' 'Á'],"{""Zé"" %!q(float32=1.89)}",%q
%+q,'*','\u00c7',"""ol\u00e1\tmundo""",['A' '\u00c1'],"{""Z\u00e9"" %!q(float32=+1.89)}",%+q
%x,2a,c7,6f6cc3a1096d756e646f,[41 c1],{5ac3a9 0x1.e3d70ap+00},%x
% x,2a,c7,6f 6c c3 a1 09 6d 75 6e 64 6f,[ 41 c1],{5a c3 a9 0x1.e3d70ap+00},% x


## Verbs for integers

```
✓ %b	base 2
✓ %c	the character represented by the corresponding Unicode code point
✓ %d	base 10
✓ %o	base 8
✓ %O	base 8 with 0o prefix
✓ %q	a single-quoted character literal safely escaped with Go syntax.
✓ %x	base 16, with lower-case letters for a-f
✓ %X	base 16, with upper-case letters for A-F
✓ %U	Unicode format: U+1234; same as "U+%04X"
```

In [4]:
%%
verbs := []string{"%T", "%#v", "%d", "%b", "%o", "%O", "%x", "%X", "%c", "%q", "%U"}
values := []any{0, -1, -1234, 10, 65, '人', 0x1f638}
gonbui.DisplayHTML(FmtTable(verbs, values))

%T,int,int,int,int,int,int32,int,%T
%#v,0,-1,-1234,10,65,20154,128568,%#v
%d,0,-1,-1234,10,65,20154,128568,%d
%b,0,-1,-10011010010,1010,1000001,100111010111010,11111011000111000,%b
%o,0,-1,-2322,12,101,47272,373070,%o
%O,0o0,-0o1,-0o2322,0o12,0o101,0o47272,0o373070,%O
%x,0,-1,-4d2,a,41,4eba,1f638,%x
%X,0,-1,-4D2,A,41,4EBA,1F638,%X
%c, ,�,�,,A,人,😸,%c
%q,'\x00','�','�','\n','A','人','😸',%q
%U,U+0000,U+FFFFFFFFFFFFFFFF,U+FFFFFFFFFFFFFB2E,U+000A,U+0041,U+4EBA,U+1F638,%U


## String and slice of bytes

(treated equivalently with these verbs):

```
✓ %s	the uninterpreted bytes of the string or slice
✓ %q	a double-quoted string safely escaped with Go syntax
✓ %x	base 16, lower-case, two characters per byte
✓ %X	base 16, upper-case, two characters per byte
```

Slice:

```
✓ %p	address of 0th element in base 16 notation, with leading 0x
```

In [5]:
%%
verbs := []string{"%T", "%#v", "%v", "%s", "%q", "%+q", "%x", "% x", "%X", "%p"}
values := []any{"A", "café", "太阳", "\U0001F638!", []byte{65, 66, 67}}
gonbui.DisplayHTML(FmtTable(verbs, values))

%T,string,string,string,string,[]uint8,%T
%#v,"""A""","""café""","""太阳""","""😸!""","[]byte{0x41, 0x42, 0x43}",%#v
%v,A,café,太阳,😸!,[65 66 67],%v
%s,A,café,太阳,😸!,ABC,%s
%q,"""A""","""café""","""太阳""","""😸!""","""ABC""",%q
%+q,"""A""","""caf\u00e9""","""\u592a\u9633""","""\U0001f638!""","""ABC""",%+q
%x,41,636166c3a9,e5a4aae998b3,f09f98b821,414243,%x
% x,41,63 61 66 c3 a9,e5 a4 aa e9 98 b3,f0 9f 98 b8 21,41 42 43,% x
%X,41,636166C3A9,E5A4AAE998B3,F09F98B821,414243,%X
%p,N/A,N/A,N/A,N/A,0x24e465922138,%p
